In [1]:
import pandas as pd
import numpy as np
import pickle
import ast
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# load data
ratings_sample = pd.read_csv('../data/processed/ratings_sample.csv')
movies = pd.read_csv('../data/processed/movies_final.csv')

with open('../models/svd_predictions.pkl', 'rb') as f:
    predicted_ratings = pickle.load(f)

with open('../models/user_id_to_idx.pkl', 'rb') as f:
    user_id_to_idx = pickle.load(f)

with open('../models/movie_encoder.pkl', 'rb') as f:
    movie_encoder = pickle.load(f)

print("Loaded!")
print("Ratings sample:", ratings_sample.shape)

Loaded!
Ratings sample: (11381066, 6)


In [3]:
# hold out 20% of ratings as test set
# the model never saw these during training
train_data, test_data = train_test_split(
    ratings_sample, 
    test_size=0.2, 
    random_state=42
)

print(f"Train size: {len(train_data):,}")
print(f"Test size: {len(test_data):,}")
print(f"\nTest sample:")
print(test_data.head(3))

Train size: 9,104,852
Test size: 2,276,214

Test sample:
          userId  movieId  rating   timestamp  user_idx  movie_idx
1853560    26930     2169     3.0   980176545      3297       1344
10227433  146190     4121     3.0  1017465556     17956       2371
10108555  144442     6078     4.0  1146485832     17757       2927


In [5]:
actual = []
predicted = []

# only evaluate on users and movies we have predictions for
for _, row in test_data.iterrows():
    user_id = row['userId']
    movie_id = row['movieId']
    actual_rating = row['rating']
    
    if user_id in user_id_to_idx and movie_id in movie_encoder.classes_:
        user_idx = user_id_to_idx[user_id]
        movie_idx = np.where(movie_encoder.classes_ == movie_id)[0][0]
        pred_rating = predicted_ratings[user_idx][movie_idx]
        
        actual.append(actual_rating)
        predicted.append(pred_rating)

actual = np.array(actual)
predicted = np.array(predicted)

rmse = np.sqrt(np.mean((actual - predicted) ** 2))
mae = np.mean(np.abs(actual - predicted))

print(f"Evaluated on {len(actual):,} ratings")
print(f"RMSE: {rmse:.4f}")
print(f"MAE:  {mae:.4f}")
print(f"\nInterpretation:")
print(f"On average your model is off by {mae:.2f} stars out of 5")

Evaluated on 2,276,214 ratings
RMSE: 0.7833
MAE:  0.5906

Interpretation:
On average your model is off by 0.59 stars out of 5


In [7]:
def precision_at_k(user_id, k=10, threshold=4.0):
    """
    Of the top K movies we recommend, what fraction does the user actually like?
    'Like' = actual rating >= threshold (4.0 out of 5)
    """
    if user_id not in user_id_to_idx:
        return None
    
    user_idx = user_id_to_idx[user_id]
    
    # get all movies this user rated in test set
    user_test = test_data[test_data['userId'] == user_id]
    if len(user_test) < k:
        return None
    
    # get predicted ratings for movies in test set only
    predictions = []
    for _, row in user_test.iterrows():
        movie_id = row['movieId']
        actual = row['rating']
        if movie_id in movie_encoder.classes_:
            movie_idx = np.where(movie_encoder.classes_ == movie_id)[0][0]
            pred = predicted_ratings[user_idx][movie_idx]
            predictions.append((movie_id, pred, actual))
    
    if len(predictions) < k:
        return None
    
    # sort by predicted rating, take top k
    predictions.sort(key=lambda x: x[1], reverse=True)
    top_k = predictions[:k]
    
    # how many of top k does user actually like?
    relevant = sum(1 for _, _, actual in top_k if actual >= threshold)
    return relevant / k

# evaluate on sample of 1000 users
sample_users = ratings_sample['userId'].unique()[:1000]
precisions = []

for user_id in sample_users:
    p = precision_at_k(user_id, k=10, threshold=4.0)
    if p is not None:
        precisions.append(p)

print(f"Precision@10: {np.mean(precisions):.4f}")
print(f"Evaluated on {len(precisions)} users")
print(f"\nInterpretation:")
print(f"On average {np.mean(precisions)*100:.1f}% of top 10 recommendations")
print(f"are movies the user actually rated 4+ stars")

Precision@10: 0.8600
Evaluated on 1000 users

Interpretation:
On average 86.0% of top 10 recommendations
are movies the user actually rated 4+ stars


In [9]:
print("=" * 50)
print("   MOVIE RECOMMENDER — EVALUATION SUMMARY")
print("=" * 50)
print()
print("Dataset:")
print(f"  Movies:          {len(movies):,}")
print(f"  Users:           {len(user_id_to_idx):,}")
print(f"  Ratings used:    {len(ratings_sample):,}")
print(f"  Test ratings:    {len(test_data):,}")
print()
print("Collaborative Filtering (SVD):")
print(f"  RMSE:            {rmse:.4f}")
print(f"  MAE:             {mae:.4f}")
print(f"  Avg error:       {mae:.2f} stars out of 5")
print()
print("Hybrid Model:")
print(f"  Precision@10:    {np.mean(precisions):.4f} ({np.mean(precisions)*100:.1f}%)")
print(f"  Users evaluated: {len(precisions):,}")
print()
print("Model Components:")
print("  ✓ Content-based filtering (TF-IDF + cosine similarity)")
print("  ✓ Collaborative filtering (SVD, k=50 latent factors)")
print("  ✓ Hybrid combination (dynamic alpha weighting)")
print("  ✓ Popularity weighting (IMDB weighted rating formula)")
print("=" * 50)

   MOVIE RECOMMENDER — EVALUATION SUMMARY

Dataset:
  Movies:          4,800
  Users:           20,000
  Ratings used:    11,381,066
  Test ratings:    2,276,214

Collaborative Filtering (SVD):
  RMSE:            0.7833
  MAE:             0.5906
  Avg error:       0.59 stars out of 5

Hybrid Model:
  Precision@10:    0.8600 (86.0%)
  Users evaluated: 1,000

Model Components:
  ✓ Content-based filtering (TF-IDF + cosine similarity)
  ✓ Collaborative filtering (SVD, k=50 latent factors)
  ✓ Hybrid combination (dynamic alpha weighting)
  ✓ Popularity weighting (IMDB weighted rating formula)
